In [1]:
from collections import defaultdict

GRID = [
    list("rsdifindthsart"),
    list("ehresodaeetgna"),
    list("netrhalxhgowip"),
    list("egedauyueaenrp"),
    list("ptnnmllmxidnee"),
    list("ohuinkthanacsm"),
    list("alnpfyldebsttn"),
    list("uumjarebemehrw"),
    list("mithdceigiugts"),
    list("tlamibftoteget"),
    list("sailniitniapen"),
    list("nstoagrniiobrt"),
    list("ietiryeesprayw"),
    list("tunenty-tessix"),
]

ROWS = len(GRID)
COLS = len(GRID[0])

DIRECTIONS = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1),
]


def normalize_text(s: str) -> str:
    """
    Lowercase and remove spaces/apostrophes/punctuation except letters and hyphen.
    Adjust this if you want to keep spaces as actual targets.
    """
    s = s.lower()
    keep = []
    for ch in s:
        if ch.isalpha() or ch == "-":
            keep.append(ch)
    return "".join(keep)


def in_bounds(r: int, c: int) -> bool:
    return 0 <= r < ROWS and 0 <= c < COLS


def find_word_paths(word: str, max_paths: int | None = None):
    """
    Find all paths for a word/phrase in the grid using 8-direction adjacency
    and no cell reuse.
    Returns a list of paths, where each path is a list of (row, col) tuples, 0-indexed.
    """
    word = normalize_text(word)
    if not word:
        return []

    results = []

    def dfs(r, c, i, used, path):
        if GRID[r][c] != word[i]:
            return

        used.add((r, c))
        path.append((r, c))

        if i == len(word) - 1:
            results.append(path[:])
            path.pop()
            used.remove((r, c))
            return

        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            if in_bounds(nr, nc) and (nr, nc) not in used:
                if GRID[nr][nc] == word[i + 1]:
                    dfs(nr, nc, i + 1, used, path)
                    if max_paths is not None and len(results) >= max_paths:
                        path.pop()
                        used.remove((r, c))
                        return

        path.pop()
        used.remove((r, c))

    starts = []
    for r in range(ROWS):
        for c in range(COLS):
            if GRID[r][c] == word[0]:
                starts.append((r, c))

    for r, c in starts:
        dfs(r, c, 0, set(), [])
        if max_paths is not None and len(results) >= max_paths:
            break

    return results


def pretty_print_path(path):
    """
    Print path as 1-indexed coordinates with letters.
    """
    out = []
    for r, c in path:
        out.append(f"{GRID[r][c].upper()}=({r+1},{c+1})")
    return " -> ".join(out)


def test_words(words):
    """
    Check a list of candidate words/phrases.
    """
    for w in words:
        paths = find_word_paths(w, max_paths=10)
        print(f"\n{w!r}: {len(paths)} path(s)")
        for idx, path in enumerate(paths[:10], 1):
            print(f"  Path {idx}: {pretty_print_path(path)}")


def load_wordlist(filepath: str, min_len: int = 5, max_len: int = 20):
    """
    Load a plain text word list: one word per line.
    """
    words = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            w = normalize_text(line.strip())
            if min_len <= len(w) <= max_len and w.isalpha():
                words.append(w)
    return words


def scan_dictionary(words, min_hits_len: int = 6, max_paths_per_word: int = 3):
    """
    Scan a dictionary and report words that appear in the grid.
    """
    hits = []
    for w in words:
        paths = find_word_paths(w, max_paths=max_paths_per_word)
        if paths and len(w) >= min_hits_len:
            hits.append((len(w), w, len(paths), paths[:max_paths_per_word]))

    hits.sort(reverse=True)
    return hits


def build_prefix_set(words):
    prefixes = set()
    for w in words:
        for i in range(1, len(w) + 1):
            prefixes.add(w[:i])
    return prefixes


def scan_all_strings_with_prefix_pruning(words, min_len: int = 6, max_len: int = 14):
    """
    More aggressive scan:
    explores all adjacency paths up to max_len and keeps only strings that are
    prefixes of some dictionary word. Good for finding candidate English fragments.

    Returns a dict: found_word -> list of paths
    """
    words = [normalize_text(w) for w in words if w]
    word_set = set(words)
    prefix_set = build_prefix_set(words)

    found = defaultdict(list)

    def dfs(r, c, used, path, current):
        if current not in prefix_set:
            return

        if len(current) >= min_len and current in word_set:
            found[current].append(path[:])

        if len(current) == max_len:
            return

        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            if in_bounds(nr, nc) and (nr, nc) not in used:
                used.add((nr, nc))
                path.append((nr, nc))
                dfs(nr, nc, used, path, current + GRID[nr][nc])
                path.pop()
                used.remove((nr, nc))

    for r in range(ROWS):
        for c in range(COLS):
            used = {(r, c)}
            path = [(r, c)]
            dfs(r, c, used, path, GRID[r][c])

    return found


    # 2) Optional dictionary scan
    # Put a word list file like words.txt next to this script.
    #
    # Example sources:
    # - wordfreq lists
    # - SCOWL word lists
    # - wordfreq top English words
    #
    # Uncomment below if you have a wordlist:
    #
    # words = load_wordlist("words.txt", min_len=5, max_len=16)
    # hits = scan_dictionary(words, min_hits_len=6)
    # print("\n=== Dictionary hits ===")
    # for length, word, count, paths in hits[:100]:
    #     print(f"{word} (len={length}, paths={count})")
    #     for p in paths:
    #         print("   ", pretty_print_path(p))

    # 3) Optional deeper prefix-pruned search
    #
    # Uncomment for a more exhaustive search:
    #
    # words = load_wordlist("words.txt", min_len=5, max_len=14)
    # found = scan_all_strings_with_prefix_pruning(words, min_len=6, max_len=14)
    # print("\n=== Prefix-pruned found words ===")
    # for word in sorted(found.keys(), key=lambda x: (-len(x), x))[:200]:
    #     print(f"{word}: {len(found[word])} path(s)")
    #     print("   ", pretty_print_path(found[word][0]))

In [54]:
candidates = ['tree', 'do', 'then', 'add', 'end','than', 'and', 'or', 'not', 'if', 'open', 'start', 'close', 'of', 'dirt']
test_words(candidates)


'tree': 7 path(s)
  Path 1: T=(3,3) -> R=(2,3) -> E=(3,2) -> E=(2,1)
  Path 2: T=(3,3) -> R=(2,3) -> E=(3,2) -> E=(4,1)
  Path 3: T=(3,3) -> R=(2,3) -> E=(3,2) -> E=(4,3)
  Path 4: T=(3,3) -> R=(3,4) -> E=(4,3) -> E=(3,2)
  Path 5: T=(11,8) -> R=(12,7) -> E=(13,7) -> E=(13,8)
  Path 6: T=(11,8) -> R=(12,7) -> E=(13,8) -> E=(13,7)
  Path 7: T=(12,14) -> R=(12,13) -> E=(11,13) -> E=(10,13)

'do': 1 path(s)
  Path 1: D=(2,7) -> O=(2,6)

'then': 3 path(s)
  Path 1: T=(3,3) -> H=(2,2) -> E=(2,1) -> N=(3,1)
  Path 2: T=(3,3) -> H=(2,2) -> E=(3,2) -> N=(3,1)
  Path 3: T=(6,7) -> H=(6,8) -> E=(7,9) -> N=(6,10)

'add': 3 path(s)
  Path 1: A=(2,8) -> D=(1,8) -> D=(2,7)
  Path 2: A=(2,8) -> D=(2,7) -> D=(1,8)
  Path 3: A=(3,6) -> D=(2,7) -> D=(1,8)

'end': 7 path(s)
  Path 1: E=(4,3) -> N=(5,3) -> D=(4,4)
  Path 2: E=(4,3) -> N=(5,4) -> D=(4,4)
  Path 3: E=(4,11) -> N=(4,12) -> D=(5,11)
  Path 4: E=(4,11) -> N=(5,12) -> D=(5,11)
  Path 5: E=(5,13) -> N=(4,12) -> D=(5,11)
  Path 6: E=(5,13) -> N=

In [21]:
import csv
with open('unigram_freq.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    words = [row[0] for row in reader if row]

print(len(words), "words loaded")

333334 words loaded


In [22]:
print("\n=== Testing candidates ===")
for idx, word in enumerate(words[:100], 1):
    paths = find_word_paths(word, max_paths=10)
    print(f"{idx}. {word!r}: {len(paths)} path(s)")
    for pidx, path in enumerate(paths[:10], 1):
        print(f"   Path {pidx}: {pretty_print_path(path)}")


=== Testing candidates ===
1. 'word': 0 path(s)
2. 'the': 10 path(s)
   Path 1: T=(1,9) -> H=(1,10) -> E=(2,9)
   Path 2: T=(1,9) -> H=(1,10) -> E=(2,10)
   Path 3: T=(2,11) -> H=(1,10) -> E=(2,9)
   Path 4: T=(2,11) -> H=(1,10) -> E=(2,10)
   Path 5: T=(3,3) -> H=(2,2) -> E=(2,1)
   Path 6: T=(3,3) -> H=(2,2) -> E=(3,2)
   Path 7: T=(6,7) -> H=(6,8) -> E=(7,9)
   Path 8: T=(7,12) -> H=(8,12) -> E=(8,11)
   Path 9: T=(7,13) -> H=(8,12) -> E=(8,11)
   Path 10: T=(9,13) -> H=(8,12) -> E=(8,11)
3. 'of': 1 path(s)
   Path 1: O=(2,6) -> F=(1,5)
4. 'and': 6 path(s)
   Path 1: A=(2,8) -> N=(1,7) -> D=(1,8)
   Path 2: A=(2,8) -> N=(1,7) -> D=(2,7)
   Path 3: A=(4,5) -> N=(5,4) -> D=(4,4)
   Path 4: A=(6,9) -> N=(6,10) -> D=(5,11)
   Path 5: A=(6,11) -> N=(5,12) -> D=(5,11)
   Path 6: A=(6,11) -> N=(6,10) -> D=(5,11)
5. 'to': 7 path(s)
   Path 1: T=(2,11) -> O=(3,11)
   Path 2: T=(5,2) -> O=(6,1)
   Path 3: T=(10,8) -> O=(10,9)
   Path 4: T=(10,10) -> O=(10,9)
   Path 5: T=(11,8) -> O=(10,9)
 

In [7]:
def find_words_from_start(start_row: int, start_col: int, words, min_len: int = 3, max_len: int = 15):
    """
    start_row, start_col are 1-indexed
    """
    r0 = start_row - 1
    c0 = start_col - 1

    if not in_bounds(r0, c0):
        raise ValueError("Start coordinate out of bounds")

    word_set = set(words)
    prefix_set = build_prefix_set(words)
    found = defaultdict(list)

    def dfs(r, c, used, path, current):
        if current not in prefix_set:
            return

        if len(current) >= min_len and current in word_set:
            found[current].append(path[:])

        if len(current) == max_len:
            return

        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            if in_bounds(nr, nc) and (nr, nc) not in used:
                used.add((nr, nc))
                path.append((nr, nc))
                dfs(nr, nc, used, path, current + GRID[nr][nc])
                path.pop()
                used.remove((nr, nc))

    used = {(r0, c0)}
    path = [(r0, c0)]
    dfs(r0, c0, used, path, GRID[r0][c0])

    return found

In [48]:
found = find_words_from_start(7, 10, words[:20000], min_len=3, max_len=12)

for word in sorted(found.keys(), key=lambda w: (-len(w), w)):
    print(f"\n{word} ({len(found[word])} path(s))")
    for path in found[word][:5]:
        print("  ", pretty_print_path(path))


banning (1 path(s))
   B=(7,10) -> A=(6,11) -> N=(5,12) -> N=(4,12) -> I=(3,13) -> N=(2,13) -> G=(2,12)

banned (1 path(s))
   B=(7,10) -> A=(6,11) -> N=(5,12) -> N=(4,12) -> E=(4,11) -> D=(5,11)

banner (1 path(s))
   B=(7,10) -> A=(6,11) -> N=(5,12) -> N=(4,12) -> E=(5,13) -> R=(4,13)

beanie (2 path(s))
   B=(7,10) -> E=(7,9) -> A=(6,9) -> N=(6,10) -> I=(5,10) -> E=(4,9)
   B=(7,10) -> E=(7,9) -> A=(6,9) -> N=(6,10) -> I=(5,10) -> E=(4,11)

badly (1 path(s))
   B=(7,10) -> A=(6,9) -> D=(7,8) -> L=(7,7) -> Y=(7,6)

bates (1 path(s))
   B=(7,10) -> A=(6,11) -> T=(7,12) -> E=(8,11) -> S=(7,11)

baths (1 path(s))
   B=(7,10) -> A=(6,11) -> T=(7,12) -> H=(8,12) -> S=(7,11)

beans (1 path(s))
   B=(7,10) -> E=(7,9) -> A=(6,9) -> N=(6,10) -> S=(7,11)

beige (1 path(s))
   B=(7,10) -> E=(8,11) -> I=(9,10) -> G=(9,9) -> E=(8,9)

baht (1 path(s))
   B=(7,10) -> A=(6,9) -> H=(6,8) -> T=(6,7)

band (3 path(s))
   B=(7,10) -> A=(6,9) -> N=(6,10) -> D=(5,11)
   B=(7,10) -> A=(6,11) -> N=(5,12) -